# Étape 2 - Partie 2 : Moteur de Recherche (Stratégie Question-Réponse)
**Objectif :** Pour chaque question du jeu de test (`test_unique`), trouver les $k=10$ réponses les plus pertinentes dans la base de connaissances (`train_unique`) en utilisant la similarité cosinus.
**Méthodes :** TF-IDF et Word2Vec.

In [10]:
# !pip install nltk sentence-transformers

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer

import nltk
import ssl

# --- Problème SSL ---
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
# ---------------------------


nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
import string

# Chargement automatique des fichiers
try:
    df_train = pd.read_csv('train_unique_e2p2.csv')
    df_test = pd.read_csv('test_unique_e2p2.csv')
except:
    df_train = pd.read_csv('data/train_unique_e2p2.csv')
    df_test = pd.read_csv('data/test_unique_e2p2.csv')

df_train['Response'] = df_train['Response'].fillna('')
df_test['Context'] = df_test['Context'].fillna('')

print(f"Données prêtes : {len(df_train)} réponses en base et {len(df_test)} questions de test.")

Données prêtes : 2742 réponses en base et 10 questions de test.


In [11]:
# ==========================================
# FONCTION DE NETTOYAGE COMMUNE AU GROUPE
# ==========================================
stop_words = set(stopwords.words('english'))

def text_process(mess):
    lower_mess = mess.lower()
    # Suppression de la ponctuation
    nopunc = [char for char in lower_mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    # Suppression des stop words et séparation en mots
    clean_mess = [word for word in nopunc.split() if word not in stop_words]
    return clean_mess

## Méthode 1 : Vectorisation par TF-IDF
Le TF-IDF donne un poids aux mots. On vectorise les réponses de l'entraînement pour créer notre "base de recherche", puis on vectorise les questions du test dans le même espace mathématique pour calculer la distance.

In [12]:
k = 3
# Utilisation de l'analyseur d'Idir
vectorizer = TfidfVectorizer(analyzer=text_process)

# Transformation en matrices mathématiques
X_train = vectorizer.fit_transform(df_train['Response'])
X_test = vectorizer.transform(df_test['Context'])

print("--- RÉSULTATS DU MOTEUR DE RECHERCHE : TF-IDF ---\n")

for i in range(len(df_test)):
    similarites = cosine_similarity(X_test[i], X_train).flatten()
    indices_top_k = similarites.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} : \"{df_test.iloc[i]['Context'][:80]}...\"")
    for rank, idx in enumerate(indices_top_k):
        print(f"   Rank {rank+1} -> [INDEX TRAIN : {idx}] | Score: {similarites[idx]:.4f}")
        print(f"   Texte : {df_train.iloc[idx]['Response'][:300]}...\n")
    print("-" * 50)

--- RÉSULTATS DU MOTEUR DE RECHERCHE : TF-IDF ---

QUESTION TEST 1 : "My mother takes care of niece whom my sister abandoned. She calls me every day c..."
   Rank 1 -> [INDEX TRAIN : 1347] | Score: 0.3401
   Texte : It is understandable that it's very hard for you to hear daily complaints from your mother regarding the caregiving of your niece. You cannot change your mother's feelings and responsibilities, which could create feelings of frustration and helplessness. It must be equally hard for your mother to as...

   Rank 2 -> [INDEX TRAIN : 1358] | Score: 0.2765
   Texte : I just want to understand before I answer. Who exactly is complaining?...

   Rank 3 -> [INDEX TRAIN : 2556] | Score: 0.2244
   Texte : It is difficult to implement healthy boundaries when the person is a parent or family member. I would encourage you to identify how it makes you feel after talking with your mother. Work on establishing healthy boundaries where you do not feel obligated to engage the complaining da

## Méthode 2 : Vectorisation Sémantique par Word2Vec
Ici, l'IA essaie de comprendre le "sens" des phrases. On transforme chaque mot en coordonnées spatiales, puis on fait la moyenne de ces coordonnées pour obtenir le vecteur global de la phrase.

In [13]:
# Préparation des phrases avec le nettoyage
phrases_train = df_train['Response'].apply(text_process).tolist()
phrases_test = df_test['Context'].apply(text_process).tolist()

# Entraînement du modèle Word2Vec
w2v_model = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=1)

# Fonction pour obtenir le vecteur moyen d'une phrase
def get_sentence_vector(words, model):
    vectors = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

vect_train = np.array([get_sentence_vector(p, w2v_model) for p in phrases_train])
vect_test = np.array([get_sentence_vector(p, w2v_model) for p in phrases_test])

print("--- RÉSULTATS DU MOTEUR DE RECHERCHE : WORD2VEC ---\n")

for i in range(len(df_test)):
    sims = cosine_similarity(vect_test[i].reshape(1, -1), vect_train).flatten()
    indices = sims.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} : \"{df_test.iloc[i]['Context'][:80]}...\"")
    for rank, idx in enumerate(indices):
        print(f"   Rank {rank+1} -> [INDEX TRAIN : {idx}] | Score: {sims[idx]:.4f}")
        print(f"   Texte : {df_train.iloc[idx]['Response'][:300]}...\n")
    print("-" * 50)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


--- RÉSULTATS DU MOTEUR DE RECHERCHE : WORD2VEC ---

QUESTION TEST 1 : "My mother takes care of niece whom my sister abandoned. She calls me every day c..."
   Rank 1 -> [INDEX TRAIN : 752] | Score: 0.9996
   Texte : As far as I can tell, you received unwanted attention, but you didn't do anything wrong.  What did your instructor say? Anything? If the outfit was not appropriate then the instructor should tell you--If he/she didn't then assume the swimsuit was okay, but the gentleman in the class wanted your atte...

   Rank 2 -> [INDEX TRAIN : 1000] | Score: 0.9996
   Texte : Hi Louisiana, You got it right...he's "supposed to be" your father. It's tough enough being adopted (unless I'm reading it wrong, I think you're adopted); what you don't need is to be verbally abused by someone who's supposed to love and protect you. I don't know how old you are (past teen years tho...

   Rank 3 -> [INDEX TRAIN : 2163] | Score: 0.9994
   Texte : Hi Cleveland, I think I get what you're feeling. Yo

## Méthode 3 : Vectorisation Contextuelle par BERT (Sentence-BERT)
BERT est un modèle d'Intelligence Artificielle de type "Transformer". Contrairement à Word2Vec qui regarde les mots un par un, BERT lit la phrase entière dans les deux sens (bidirectionnel) pour comprendre le contexte exact de chaque mot avant de générer le vecteur mathématique.

In [14]:
# On utilise Sentence-BERT pour le clustering de phrases
model_bert = SentenceTransformer('all-MiniLM-L6-v2')

# Encodage (BERT travaille mieux sur le texte brut ou légèrement nettoyé)
# On utilise 'Response_clean' si disponible ou juste le texte
print("Encodage BERT en cours...")
bert_train = model_bert.encode(df_train['Response'].tolist())
bert_test = model_bert.encode(df_test['Context'].tolist())

print("\n--- RÉSULTATS DU MOTEUR DE RECHERCHE : BERT ---\n")

for i in range(len(df_test)):
    sims_bert = cosine_similarity(bert_test[i].reshape(1, -1), bert_train).flatten()
    indices_bert = sims_bert.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} : \"{df_test.iloc[i]['Context'][:80]}...\"")
    for rank, idx in enumerate(indices_bert):
        print(f"   Rank {rank+1} -> [INDEX TRAIN : {idx}] | Score: {sims_bert[idx]:.4f}")
        print(f"   Texte : {df_train.iloc[idx]['Response'][:300]}...\n")
    print("-" * 50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10963.41it/s]


Encodage BERT en cours...

--- RÉSULTATS DU MOTEUR DE RECHERCHE : BERT ---

QUESTION TEST 1 : "My mother takes care of niece whom my sister abandoned. She calls me every day c..."
   Rank 1 -> [INDEX TRAIN : 1347] | Score: 0.7216
   Texte : It is understandable that it's very hard for you to hear daily complaints from your mother regarding the caregiving of your niece. You cannot change your mother's feelings and responsibilities, which could create feelings of frustration and helplessness. It must be equally hard for your mother to as...

   Rank 2 -> [INDEX TRAIN : 2556] | Score: 0.5907
   Texte : It is difficult to implement healthy boundaries when the person is a parent or family member. I would encourage you to identify how it makes you feel after talking with your mother. Work on establishing healthy boundaries where you do not feel obligated to engage the complaining daily. Maybe setting...

   Rank 3 -> [INDEX TRAIN : 1348] | Score: 0.5591
   Texte : Then one day when life betw